EXPLORACIÓN DE DATOS, GRÁFICAS Y DEMÁS -> MACARENA Y MIGUEL

# 02 — Enrichment

## [ENG] Objectives

Walk through the enrichment process step by step:

1. **Merge sensors** — Combine VIIRS + MODIS into a single dataset
2. **Confidence mapping** — Reconcile numerical (VIIRS) and categorical (MODIS) confidence
3. **CLC enrichment** — Add land cover class to each fire detection
4. **Weather enrichment** — Add historical weather data to each fire detection

Each step calls functions from `src/wildfire/` and shows the resulting DataFrame.

## [ESP] Objetivos

Recorrer el proceso de enriquecimiento paso a paso:

1. **Fusión de sensores** — Combinar VIIRS + MODIS en un solo dataset
2. **Mapeo de confianza** — Reconciliar confianza numérica (VIIRS) y categórica (MODIS)
3. **Enriquecimiento CLC** — Añadir clase de cobertura del suelo a cada detección
4. **Enriquecimiento climático** — Añadir datos de clima histórico a cada detección

Cada paso llama funciones de `src/wildfire/` y muestra el DataFrame resultante.

In [6]:
import pandas as pd
import numpy as np
import holidays

df = pd.read_csv(r'C:\Users\Macarena\OneDrive\Documentos\wildfires-project\data\processed\enriched\firms_spain_final.csv')

# DESCOMPOSICIÓN TEMPORAL
df['acq_date'] = pd.to_datetime(df['acq_date'])

df['mes'] = df['acq_date'].dt.month
df['dia_del_año'] = df['acq_date'].dt.dayofyear
df['es_fin_semana'] = df['acq_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Senos y cosenos
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df['dia_año_sin'] = np.sin(2 * np.pi * df['dia_del_año'] / 365.25)
df['dia_año_cos'] = np.cos(2 * np.pi * df['dia_del_año'] / 365.25)

# Festivos 
mapa_ccaa_holidays = {
    'Andalucía': 'AN', 'Aragón': 'AR', 'Asturias': 'AS', 'Islas Baleares': 'IB', 
    'Canarias': 'CN', 'Cantabria': 'CB', 'Castilla y León': 'CL', 
    'Castilla-La Mancha': 'CM', 'Cataluña': 'CT', 'Comunidad Valenciana': 'VC', 
    'Extremadura': 'EX', 'Galicia': 'GA', 'Madrid': 'MD', 
    'Región de Murcia': 'MC', 'Navarra': 'NC', 'País Vasco': 'PV', 'La Rioja': 'RI'
}

# Calcular festivos solo para combinaciones únicas de fecha/CCAA 
calendario_unico = df[['acq_date', 'ccaa']].dropna().drop_duplicates()

def evaluar_festivo(fila):
    codigo_region = mapa_ccaa_holidays.get(fila['ccaa'])
    if not codigo_region:
        return 0
    cal = holidays.Spain(years=fila['acq_date'].year, subdiv=codigo_region)
    return int(fila['acq_date'] in cal)

calendario_unico['es_festivo'] = calendario_unico.apply(evaluar_festivo, axis=1)

df = df.merge(calendario_unico, on=['acq_date', 'ccaa'], how='left')
df['es_festivo'] = df['es_festivo'].fillna(0).astype(int)

# Variable riesgo antrópico
df['riesgo_antropico_alto'] = ((df['es_fin_semana'] == 1) | (df['es_festivo'] == 1)).astype(int)

# Franjas horarias
df['acq_time_str'] = df['acq_time'].astype(str).str.zfill(4)
df['hora'] = pd.to_datetime(df['acq_time_str'], format='%H%M', errors='coerce').dt.hour

bins_hora = [-1, 6, 12, 18, 24]
labels_hora = ['Madrugada', 'Mañana', 'Tarde_Pico_Calor', 'Noche']
df['franja_horaria'] = pd.cut(df['hora'], bins=bins_hora, labels=labels_hora, right=False)

# Hectáreas forestales (MITECO)
superficie_forestal_ha = {
    'Andalucía': 4400000, 'Aragón': 2600000, 'Asturias': 770000,
    'Islas Baleares': 220000, 'Canarias': 570000, 'Cantabria': 360000,
    'Castilla y León': 5100000, 'Castilla-La Mancha': 3600000, 'Cataluña': 2000000,
    'Comunidad Valenciana': 1300000, 'Extremadura': 2700000, 'Galicia': 2000000,
    'Madrid': 430000, 'Región de Murcia': 510000, 'Navarra': 610000,
    'País Vasco': 490000, 'La Rioja': 340000
}

df['sup_forestal_ha'] = df['ccaa'].map(superficie_forestal_ha)

df['ratio_prevencion_extincion'] = df['sum_prevention'] / (df['sum_extinction'] + 1e-5)
df['euros_prevencion_por_ha'] = (df['sum_prevention'] * 1000000) / df['sup_forestal_ha']
df['euros_extincion_por_ha'] = (df['sum_extinction'] * 1000000) / df['sup_forestal_ha']


# Potencial de secado atmosférico acelerado por el viento
df['potencial_propagacion'] = df['vapour_pressure_deficit'] * df['wind_speed_10m']

df['indice_riesgo_meteo'] = df['potencial_propagacion'] / (df['soil_moisture_0_to_7cm'] + 0.01)

df['estres_termico_vegetacion'] = df['temperature_2m'] / (df['relative_humidity_2m'] + 1)


df['indice_riesgo_meteo_log'] = np.log1p(df['indice_riesgo_meteo'])

# 7. Limpieza final
columnas_basura = [
    'acq_time_str', 'hora', 'sup_forestal_ha', 
    'es_fin_semana', 'es_festivo'
]
df = df.drop(columns=columnas_basura)